# Semantic Entropy for Uncertainty Quantification in LLM Outputs

**Sheroz Khan** | iRisk Lab, UIUC | Fall 2026

## Week 1: Paper Study and Proof-of-Concept Implementation

**Paper:** Farquhar, S., Kossen, J., Kuhn, L., and Gal, Y. (2024). "Detecting Hallucinations in Large Language Models Using Semantic Entropy." *Nature*, 630, 625-630. https://doi.org/10.1038/s41586-024-07421-0

(Earlier version: Kuhn, Gal, and Farquhar, ICLR 2023, arXiv:2302.09664. Same method; the Nature version is the extended, final publication.)

---

## Research Question

Can semantic entropy, which measures uncertainty over meaning clusters rather than token sequences, reliably distinguish correct from incorrect LLM outputs on factual questions?

This notebook implements the semantic entropy pipeline from Farquhar et al. (*Nature*, 2024) from scratch and evaluates it on a pilot set of questions using a local LLM via Ollama.

## Motivation

Standard token-level uncertainty (e.g., predictive entropy over the next-token distribution) conflates two sources of variation:

1. **Semantic uncertainty:** the model is unsure *what* to say (different meanings).
2. **Lexical/syntactic uncertainty:** the model is unsure *how to phrase* a known answer.

For applications where the meaning of the output matters more than the exact wording, token-level entropy is misleading. The same correct answer phrased differently looks like disagreement to a token-level metric. Semantic entropy resolves this by clustering generations by meaning and computing entropy over those clusters.

**Example:** For the question "What is the capital of France?", the responses "Paris", "The capital is Paris", and "It's Paris" all mean the same thing. Token-level entropy sees three different strings and reports high uncertainty. Semantic entropy sees one meaning cluster and reports zero uncertainty.

## Paper Summary

**What the paper does:** Proposes semantic entropy as an uncertainty measure for LLM outputs. The method samples multiple generations, clusters them by semantic equivalence using a natural language inference (NLI) model, and computes Shannon entropy over the resulting cluster distribution.

**Key result:** High semantic entropy correlates with confabulation (the model generates confident-sounding but semantically inconsistent outputs) on TriviaQA, SQuAD, and BioASQ benchmarks. Outperforms token-level entropy, predictive entropy, and self-consistency baselines.

## Mathematical Formulation

### Step 1: Generation Probability (Equation 1)

Given input query $x$, each sampled generation $s = (t_1, t_2, \ldots, t_n)$ has probability under the standard autoregressive factorization:

$$p(s \mid x) = \prod_{i=1}^{n} p(t_i \mid t_{1:i-1}, x)$$

Each factor $p(t_i \mid t_{1:i-1}, x)$ is the softmax probability the LLM assigns to token $t_i$ given the query $x$ and all preceding tokens.

### Step 2: Semantic Clustering via Bidirectional NLI Entailment

Two generations $s_a$ and $s_b$ are defined as **semantically equivalent** if and only if:

$$s_a \Rightarrow s_b \quad \text{AND} \quad s_b \Rightarrow s_a$$

where $\Rightarrow$ denotes textual entailment as classified by a DeBERTa NLI model (trained on MNLI).

A **greedy algorithm** assigns each generation to clusters:
- For each generation $s_j$: check if it bidirectionally entails a representative of any existing cluster.
- If yes, assign $s_j$ to that cluster.
- If no, create a new cluster containing only $s_j$.

This produces $K$ clusters $(C_1, \ldots, C_K)$.

**Caveat:** Bidirectional NLI entailment is not a true equivalence relation. Entailment is not guaranteed to be transitive. The greedy algorithm is also order-dependent.

### Step 3: Cluster Probability

**Regular variant** (white-box, requires token probabilities):

$$p(C_k \mid x) = \sum_{s \in C_k} p(s \mid x)$$

In practice this is intractable (requires summing over all possible sequences in the cluster). The paper approximates by summing only over sampled generations.

**Discrete variant** (black-box compatible, what the paper mostly reports):

$$p(C_k \mid x) = \frac{1}{N} \sum_{j=1}^{N} \mathbb{1}[s_j \in C_k]$$

Simply the fraction of $N$ sampled generations in cluster $k$. Avoids sequence-length bias and does not require token probabilities.

### Step 4: Semantic Entropy (Equation 3)

Shannon entropy over the cluster distribution:

$$H_{\text{SE}}(x) = -\sum_{k=1}^{K} p(C_k \mid x) \cdot \log p(C_k \mid x)$$

**Interpretation:**
- $H_{\text{SE}} = 0$: all generations in one meaning cluster (semantically certain)
- $H_{\text{SE}} = \log K$: uniform spread across $K$ clusters (maximum uncertainty)

## What the Paper Claims vs. What It Actually Does

### Claim: "Detects hallucinations"
**Reality:** Flags high semantic entropy, which correlates with confabulation on short-form QA benchmarks. Does not verify whether any particular answer is correct. Measures semantic consistency across samples, not correctness.

### Claim: "Linguistic invariance"
**Reality:** The invariance comes from the NLI clustering step. Its quality depends entirely on DeBERTa-MNLI. If the NLI model fails to recognize two answers as equivalent, they get split into separate clusters and SE is inflated. Not tested on domain-specific text.

### Claim: "Outperforms predictive entropy"
**Reality:** On short-form factoid QA (TriviaQA, SQuAD, BioASQ) only. Long-form generation, numerical outputs, and multi-step reasoning are not evaluated.

### Approximations not emphasized in the paper
1. **N=10 samples.** Entropy estimate has high variance. Rare clusters can be missed.
2. **Greedy clustering is order-dependent.** Different sample orderings can produce different clusters.
3. **Discrete variant discards probability information.** A high-probability cluster and a low-probability cluster with equal sample counts get equal weight.

## Environment Setup

### Requirements
```bash
pip install transformers torch numpy pandas matplotlib requests
```

### Ollama Setup
This notebook uses a local LLM via [Ollama](https://ollama.com). Install Ollama and pull the model:
```bash
# Install Ollama (https://ollama.com/download)
# Then pull the model:
ollama pull gemma3:4b
```
Verify it is running:
```bash
ollama list
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import math
import json
import time
import warnings
warnings.filterwarnings('ignore')

print("Base packages loaded.")

## Implementation: NLI Model

DeBERTa-large fine-tuned on MNLI, following Farquhar et al. (2024). Runs on CPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

NLI_MODEL_NAME = "microsoft/deberta-large-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.eval()

NLI_LABELS = {0: "contradiction", 1: "neutral", 2: "entailment"}

print(f"NLI model loaded: {NLI_MODEL_NAME}")
print(f"Label mapping: {NLI_LABELS}")

## Implementation: Bidirectional Entailment Check

For two generations $s_a$ and $s_b$, check entailment in both directions. Semantically equivalent only if both directions yield "entailment." Question context is prepended to help the NLI model interpret short answers.

In [ ]:
def check_entailment(premise: str, hypothesis: str) -> str:
    """Classify the NLI relation: entailment, contradiction, or neutral."""
    inputs = nli_tokenizer(
        premise, hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        logits = nli_model(**inputs).logits
    pred = logits.argmax(dim=-1).item()
    return NLI_LABELS[pred]


def are_semantically_equivalent(s_a: str, s_b: str, context: str = "") -> bool:
    """Bidirectional entailment check, following Farquhar et al. (2024)."""
    if context:
        s_a_ctx = f"Question: {context} Answer: {s_a}"
        s_b_ctx = f"Question: {context} Answer: {s_b}"
    else:
        s_a_ctx, s_b_ctx = s_a, s_b

    forward = check_entailment(s_a_ctx, s_b_ctx)
    backward = check_entailment(s_b_ctx, s_a_ctx)

    return (forward == "entailment") and (backward == "entailment")


# === Verification ===
print("--- Verification ---")
ctx = "What is the capital of France?"

pairs = [
    ("Paris", "The capital of France is Paris", True),
    ("Paris", "It's Paris", True),
    ("Paris", "London", False),
]

for a, b, expected in pairs:
    result = are_semantically_equivalent(a, b, context=ctx)
    status = "OK" if result == expected else "MISMATCH"
    print(f"  '{a}' vs '{b}': {result} (expected {expected}) [{status}]")

## Implementation: Greedy Semantic Clustering

Greedy algorithm from the paper: each generation is compared against the representative (first member) of each existing cluster.

In [ ]:
def cluster_by_meaning(generations: list, context: str = "") -> list:
    """
    Greedy semantic clustering via bidirectional NLI entailment.
    Returns: list of lists, each inner list is a semantic cluster.
    """
    clusters = []

    for gen in generations:
        assigned = False
        for cluster in clusters:
            representative = cluster[0]
            if are_semantically_equivalent(gen, representative, context):
                cluster.append(gen)
                assigned = True
                break

        if not assigned:
            clusters.append([gen])

    return clusters

## Implementation: Discrete Semantic Entropy

$$H_{\text{SE}}(x) = -\sum_{k=1}^{K} \frac{|C_k|}{N} \cdot \log \frac{|C_k|}{N}$$

In [ ]:
def compute_semantic_entropy(clusters: list, n_total: int) -> float:
    """Discrete semantic entropy (Shannon entropy over cluster fractions, in nats)."""
    entropy = 0.0
    for cluster in clusters:
        p_k = len(cluster) / n_total
        if p_k > 0:
            entropy -= p_k * math.log(p_k)
    return entropy


# === Worked example ===
# All same meaning: expect SE = 0
test_clusters_same = [["Paris", "The capital is Paris", "It's Paris", "Paris", "Paris"]]
print(f"All same meaning: SE = {compute_semantic_entropy(test_clusters_same, 5):.4f} (expect 0)")

# Split 4/3/3: expect SE near log(3)
test_clusters_split = [["a"]*4, ["b"]*3, ["c"]*3]
se_split = compute_semantic_entropy(test_clusters_split, 10)
print(f"Split 4/3/3:      SE = {se_split:.4f} (max for K=3: {math.log(3):.4f})")

## LLM Generation via Ollama (Local Gemma)

Generates N samples from a local Gemma model running through Ollama. The Ollama API runs at `http://localhost:11434` by default.

**Temperature is set to 1.0** to produce diverse samples, following the paper.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "gemma3:4b"  # Change to your pulled model name

def generate_samples_ollama(
    question: str,
    n_samples: int = 10,
    model: str = OLLAMA_MODEL,
    temperature: float = 1.0,
    max_tokens: int = 100
) -> list:
    """
    Generate n_samples responses from a local Ollama model.
    Returns a list of response strings.
    """
    system_prompt = "Answer the question concisely. Give only the answer, no explanation or reasoning."
    prompt = f"{system_prompt}\n\nQuestion: {question}\nAnswer:"

    generations = []
    for i in range(n_samples):
        try:
            resp = requests.post(OLLAMA_URL, json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": temperature,
                    "num_predict": max_tokens,
                }
            }, timeout=120)
            resp.raise_for_status()
            text = resp.json().get("response", "").strip()
            # Clean: take first line only (avoid rambling)
            text = text.split("\n")[0].strip()
            generations.append(text)
        except Exception as e:
            print(f"  Sample {i+1} failed: {e}")
            generations.append("")

    return generations


# === Test connection ===
def test_ollama_connection():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama is running. Available models: {models}")
        return True
    except Exception as e:
        print(f"Ollama not reachable: {e}")
        print("Start Ollama and run: ollama pull gemma3:4b")
        return False

ollama_available = test_ollama_connection()

In [ ]:
# === Pilot questions ===
PILOT_QUESTIONS = [
    {"question": "What is the capital of France?", "answer": "Paris"},
    {"question": "Who painted the Mona Lisa?", "answer": "Leonardo da Vinci"},
    {"question": "What is the smallest planet in our solar system?", "answer": "Mercury"},
    {"question": "What is the chemical formula for table salt?", "answer": "NaCl"},
    {"question": "In what year did the Titanic sink?", "answer": "1912"},
    {"question": "Who developed the theory of general relativity?", "answer": "Albert Einstein"},
    {"question": "What is the largest organ in the human body?", "answer": "Skin"},
    {"question": "What is the speed of light in meters per second?", "answer": "3 x 10^8"},
    {"question": "What is the powerhouse of the cell?", "answer": "Mitochondria"},
    {"question": "Who wrote Romeo and Juliet?", "answer": "William Shakespeare"},
]

# === Precomputed fallback (if Ollama is not available) ===
PRECOMPUTED = {
    "What is the capital of France?": [
        "Paris", "The capital of France is Paris.", "Paris.", "Paris is the capital of France.",
        "Paris", "It's Paris.", "Paris", "Paris.", "Paris", "Paris."],
    "Who painted the Mona Lisa?": [
        "Leonardo da Vinci", "The Mona Lisa was painted by Leonardo da Vinci.",
        "Leonardo da Vinci.", "Da Vinci", "Leonardo da Vinci",
        "Leonardo da Vinci painted it.", "Leonardo da Vinci",
        "It was Leonardo da Vinci.", "Leonardo da Vinci", "Da Vinci."],
    "What is the smallest planet in our solar system?": [
        "Mercury", "The smallest planet is Mercury.", "Mercury.",
        "Mercury is the smallest.", "Pluto", "Mercury", "Mercury.", "Mercury", "Mercury", "Mercury"],
    "What is the chemical formula for table salt?": [
        "NaCl", "NaCl", "The chemical formula is NaCl.", "NaCl", "NaCl",
        "Sodium chloride, or NaCl.", "NaCl", "NaCl", "NaCl", "NaCl"],
    "In what year did the Titanic sink?": [
        "1912", "The Titanic sank in 1912.", "1912.", "1912", "1912",
        "1912", "1912", "It sank in 1912.", "1912", "1912"],
    "Who developed the theory of general relativity?": [
        "Albert Einstein", "Einstein", "Albert Einstein.", "Albert Einstein developed it.",
        "Einstein", "Albert Einstein", "Einstein.", "Albert Einstein", "Albert Einstein",
        "It was Albert Einstein."],
    "What is the largest organ in the human body?": [
        "The skin", "Skin", "The skin.", "Skin", "The liver",
        "Skin", "The skin is the largest organ.", "Skin", "Skin.", "Skin"],
    "What is the speed of light in meters per second?": [
        "Approximately 3 x 10^8 m/s", "299,792,458 m/s", "3 x 10^8 m/s",
        "About 300,000 km/s", "299,792,458 meters per second", "3 x 10^8 m/s",
        "~3 x 10^8 m/s", "The speed of light is approximately 3 x 10^8 m/s.",
        "299792458 m/s", "3 x 10^8 m/s"],
    "What is the powerhouse of the cell?": [
        "The mitochondria", "Mitochondria", "The mitochondria.",
        "Mitochondria is the powerhouse of the cell.", "The mitochondria",
        "Mitochondria", "Mitochondria.", "The mitochondria", "Mitochondria", "Mitochondria"],
    "Who wrote Romeo and Juliet?": [
        "William Shakespeare", "Shakespeare", "William Shakespeare.",
        "Shakespeare wrote Romeo and Juliet.", "William Shakespeare", "Shakespeare.",
        "William Shakespeare", "It was William Shakespeare.", "William Shakespeare", "Shakespeare"],
}


def get_samples(question: str, n: int = 10) -> tuple:
    """Returns (samples, source) where source is 'ollama' or 'precomputed'."""
    if ollama_available:
        samples = generate_samples_ollama(question, n_samples=n)
        # Filter empty responses
        samples = [s for s in samples if s]
        if len(samples) >= 5:  # Need at least 5 usable samples
            return samples, "ollama"

    # Fallback to precomputed
    return PRECOMPUTED.get(question, [])[:n], "precomputed"


print(f"Generation source: {'Ollama (' + OLLAMA_MODEL + ')' if ollama_available else 'Precomputed fallback'}")
print(f"Pilot questions: {len(PILOT_QUESTIONS)}")

## Pilot Experiment

For each question:
1. Generate 10 samples (Ollama if available, precomputed fallback otherwise)
2. Cluster by bidirectional NLI entailment
3. Compute discrete semantic entropy
4. Check correctness of majority answer

In [ ]:
results = []

for item in PILOT_QUESTIONS:
    question = item["question"]
    correct_answer = item["answer"]

    print(f"Q: {question}")

    samples, source = get_samples(question)
    n = len(samples)

    if n == 0:
        print("  SKIPPED: no samples available")
        continue

    # Cluster
    clusters = cluster_by_meaning(samples, context=question)

    # Semantic entropy
    se = compute_semantic_entropy(clusters, n)

    # Majority answer
    answer_counts = Counter(samples)
    majority = answer_counts.most_common(1)[0][0]

    # Correctness
    is_correct = correct_answer.lower() in majority.lower()

    results.append({
        "question": question,
        "correct_answer": correct_answer,
        "n_samples": n,
        "n_clusters": len(clusters),
        "semantic_entropy": round(se, 4),
        "majority_answer": majority,
        "is_correct": is_correct,
        "source": source,
        "clusters": clusters,
        "raw_samples": samples,
    })

    print(f"  Source: {source} | Clusters: {len(clusters)} | SE: {se:.4f} | Correct: {is_correct}")
    for i, c in enumerate(clusters):
        print(f"    C{i+1}: {c}")
    print()

## Results

In [ ]:
results_df = pd.DataFrame([{
    "Question": r["question"],
    "K (clusters)": r["n_clusters"],
    "SE (nats)": r["semantic_entropy"],
    "Correct": r["is_correct"],
    "Source": r["source"],
} for r in results])

display(results_df)

correct_se = results_df[results_df['Correct']]['SE (nats)']
incorrect_se = results_df[~results_df['Correct']]['SE (nats)']

print(f"\nCorrect answers   - Mean SE: {correct_se.mean():.4f}, n={len(correct_se)}")
if len(incorrect_se) > 0:
    print(f"Incorrect answers - Mean SE: {incorrect_se.mean():.4f}, n={len(incorrect_se)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if c else '#e74c3c' for c in results_df['Correct']]
y_pos = range(len(results_df))

ax.barh(y_pos, results_df['SE (nats)'], color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels([q[:40] + "..." if len(q) > 40 else q for q in results_df['Question']], fontsize=9)
ax.set_xlabel("Semantic Entropy (nats)")
ax.set_title("Semantic Entropy by Question (Farquhar et al., 2024)")
ax.legend(
    handles=[
        plt.Rectangle((0,0),1,1, color='#2ecc71', label='Correct (majority)'),
        plt.Rectangle((0,0),1,1, color='#e74c3c', label='Incorrect (majority)')
    ],
    loc='lower right'
)

plt.tight_layout()
plt.savefig("semantic_entropy_pilot.png", dpi=150, bbox_inches='tight')
plt.show()

## Observations

### What the implementation confirms
- Questions where the model consistently produces the same meaning receive low or zero semantic entropy, as the paper predicts.
- The NLI clustering step successfully groups paraphrases into the same cluster, which raw string matching would not do.

### Limitations of this pilot
- **Easy questions.** The pilot uses factoid questions where LLMs are generally reliable. Harder or adversarial questions are needed to test hallucination detection.
- **Small pilot (10 questions).** Too few for statistical conclusions. This is a proof-of-concept.
- **NLI model limitations.** DeBERTa-MNLI may mishandle numerical equivalences (e.g., "3 x 10^8" vs. "299,792,458") or domain-specific text.

### Observations on the method
- **Order dependence.** The greedy clustering produces different results if samples are reordered. The paper does not resolve this.
- **Discrete variant discards probabilities.** Equal sample counts get equal weight regardless of generation probability.
- **N=10 is coarse.** Minimum nonzero cluster probability is 0.1, limiting entropy resolution.

## Next Steps

- Scale to 50+ questions including questions where the LLM is known to hallucinate.
- Compare SE against token-level predictive entropy and self-verbalized confidence.
- Test NLI clustering on domain-specific text (actuarial, insurance terminology).
- Investigate sensitivity of SE to number of samples N.

## References

Farquhar, S., Kossen, J., Kuhn, L., and Gal, Y. (2024). Detecting Hallucinations in Large Language Models Using Semantic Entropy. *Nature*, 630, 625-630. https://doi.org/10.1038/s41586-024-07421-0

Kuhn, L., Gal, Y., and Farquhar, S. (2023). Semantic Uncertainty: Linguistic Invariances for Uncertainty Estimation in Natural Language Generation. *ICLR 2023*. arXiv:2302.09664.